In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

print("Libraries Loaded Successfully")

Libraries Loaded Successfully


In [2]:
master = pd.read_csv(
    'cleaned_master_dataset.csv'
)

print(
    "Dataset Loaded Successfully"
)

Dataset Loaded Successfully


In [3]:
master[
    'order_purchase_timestamp'
] = pd.to_datetime(
    master[
        'order_purchase_timestamp'
    ],
    format='mixed',
    dayfirst=True
)

In [4]:
snapshot_date = (
    master[
        'order_purchase_timestamp'
    ].max()
    +
    pd.Timedelta(days=1)
)

customer_data = (
    master.groupby(
        'customer_unique_id'
    )
    .agg({
        'order_purchase_timestamp':'max',
        'revenue':'sum',
        'order_id':'nunique'
    })
    .reset_index()
)

customer_data.columns = [
    'customer_unique_id',
    'last_purchase',
    'total_revenue',
    'total_orders'
]

customer_data['Recency'] = (
    snapshot_date
    -
    customer_data[
        'last_purchase'
    ]
).dt.days

customer_data['Churn'] = np.where(
    customer_data['Recency'] > 90,
    1,
    0
)

customer_data.head()

,customer_unique_id,last_purchase,total_revenue,total_orders,Recency,Churn
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:00,0.0,1,161,1
1,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:00,69.0,1,586,1
2,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:00,0.0,1,337,1
3,00053a61a98854899e70ed204dd4bafe,2018-02-28 11:15:00,382.0,1,232,1
4,0005ef4cd20d2893f0d9fbd94d3c0d97,2018-03-12 15:22:00,104.9,1,220,1


In [6]:
master.shape

(82365, 39)

In [7]:
customer_data[
    'Churn'
].value_counts()

,count
Churn,
1,62305
0,6822


In [8]:
features = [
    'total_revenue',
    'total_orders',
    'Recency'
]

X = customer_data[
    features
]

y = customer_data[
    'Churn'
]

In [9]:
X_train, X_test, y_train, y_test = (
    train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )
)

In [10]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(
    X_train,
    y_train
)

print(
    "Model Trained Successfully"
)

Model Trained Successfully


In [11]:
y_pred = rf_model.predict(X_test)

print(
    "Accuracy:",
    accuracy_score(
        y_test,
        y_pred
    )
)

Accuracy: 1.0


In [12]:
print(
    classification_report(
        y_test,
        y_pred
    )
)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1299
           1       1.00      1.00      1.00     12527

    accuracy                           1.00     13826
   macro avg       1.00      1.00      1.00     13826
weighted avg       1.00      1.00      1.00     13826



In [13]:
confusion_matrix(
    y_test,
    y_pred
)

array([[ 1299,     0],
       [    0, 12527]])

In [14]:
customer_data['Churn'].value_counts(normalize=True) * 100

,proportion
Churn,
1,90.131208
0,9.868792


In [15]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42
)

In [18]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42
)

rf_model.fit(
    X_train,
    y_train
)

print("Model Trained Successfully")

Model Trained Successfully


In [19]:
features = [
    'total_revenue',
    'total_orders',
    'Recency'
]

X = customer_data[features]
y = customer_data['Churn']

In [20]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [21]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42
)

rf_model.fit(
    X_train,
    y_train
)

print("Model Trained Successfully")

Model Trained Successfully


In [22]:
feature_importance = pd.DataFrame({
    'Feature': features,
    'Importance': rf_model.feature_importances_
})

feature_importance.sort_values(
    'Importance',
    ascending=False
)

,Feature,Importance
2,Recency,0.997602
0,total_revenue,0.002036
1,total_orders,0.000362


In [23]:
print(rf_model)

RandomForestClassifier(class_weight='balanced', n_estimators=200,
                       random_state=42)


In [24]:
hasattr(rf_model, "feature_importances_")

True

In [25]:
customer_data['Churn_Probability'] = (
    rf_model.predict_proba(X)[:,1]
)

customer_data[
    [
        'customer_unique_id',
        'Churn_Probability'
    ]
].head()

,customer_unique_id,Churn_Probability
0,0000366f3b9a7992bf8c76cfdf3221e2,1.0
1,0000f46a3911fa3c0805444483337064,1.0
2,0004aac84e0df4da2b147fca70cf8255,1.0
3,00053a61a98854899e70ed204dd4bafe,1.0
4,0005ef4cd20d2893f0d9fbd94d3c0d97,1.0


In [26]:
def future_risk(prob):

    if prob >= 0.80:
        return 'Critical'

    elif prob >= 0.60:
        return 'High'

    elif prob >= 0.40:
        return 'Medium'

    else:
        return 'Low'


customer_data['Future_Risk'] = (
    customer_data[
        'Churn_Probability'
    ].apply(
        future_risk
    )
)

customer_data[
    'Future_Risk'
].value_counts()

,count
Future_Risk,
Critical,62305
Low,6822


In [27]:
future_churn_customers = (
    customer_data
    .sort_values(
        'Churn_Probability',
        ascending=False
    )
    .head(100)
)

future_churn_customers.head(10)

,customer_unique_id,last_purchase,total_revenue,total_orders,Recency,Churn,Churn_Probability,Future_Risk
69126,ffffd2657e2aad2907e67c3e9daecbeb,2017-05-02 20:18:00,0.0,1,533,1,1.0,Critical
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:00,0.0,1,161,1,1.0,Critical
1,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:00,69.0,1,586,1,1.0,Critical
2,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:00,0.0,1,337,1,1.0,Critical
3,00053a61a98854899e70ed204dd4bafe,2018-02-28 11:15:00,382.0,1,232,1,1.0,Critical
4,0005ef4cd20d2893f0d9fbd94d3c0d97,2018-03-12 15:22:00,104.9,1,220,1,1.0,Critical
5,0006fdc98a402fceb4eb0ee528f6a8d4,2017-07-18 09:23:00,13.9,1,457,1,1.0,Critical
69109,ffedff0547d809c90c05c2691c51f9b7,2017-03-30 14:50:00,0.0,1,567,1,1.0,Critical
69108,ffeddf8aa7cdecf403e77b2e9a99e2ea,2018-05-13 16:04:00,330.0,1,158,1,1.0,Critical
69106,ffec10ad4229ba46818560e1c8b40a68,2018-04-05 05:11:00,120.0,1,196,1,1.0,Critical


In [28]:
feature_importance.to_csv(
    'feature_importance.csv',
    index=False
)

print("Feature Importance Saved")

Feature Importance Saved


In [29]:
customer_data.to_csv(
    'customer_churn_predictions.csv',
    index=False
)

print("Customer Churn Predictions Saved")

Customer Churn Predictions Saved


In [30]:
future_churn_customers.to_csv(
    'future_churn_customers.csv',
    index=False
)

print("Future Churn Customers Saved")

Future Churn Customers Saved


In [31]:
performance = pd.DataFrame({
    'Metric': [
        'Accuracy'
    ],
    'Value': [
        accuracy_score(
            y_test,
            y_pred
        )
    ]
})

performance.to_csv(
    'model_performance.csv',
    index=False
)

performance

,Metric,Value
0,Accuracy,1.0


In [32]:
import os

os.listdir()

['.config',
 'customer_churn_predictions.csv',
 'cleaned_master_dataset.csv',
 'model_performance.csv',
 'feature_importance.csv',
 'future_churn_customers.csv',
 'sample_data']